## Deploying a CPLEX export file with MIP starts with watsonx.ai Runtime

This notebook shows you how to deploy a CPLEX export file with MIP starts, create and monitor jobs, and get solution and logs using the watsonx.ai Python Client.

This notebook runs on Python.

**Table of contents:**

- [Set up the watsonx.ai client](#setup)
- [Create a client instance](#create)
- [Prepare your model archive](#prepare)
- [Upload your model on watsonx.ai Runtime](#upload)
- [Create a deployment](#deploy)
- [Create and monitor a job with inline data for your deployed model](#job)
- [Summary](#summary)

<a id='setup'></a>
### Set up the watsonx.ai client

Before you use the sample code in this notebook, you must:

- create a <a href="https://cloud.ibm.com/catalog?category=ai" target="_blank" rel="noopener noreferrer">watsonx.ai Runtime Service</a> instance. A free plan is offered and information about how to create the instance can be found at <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas" target="_blank" rel="noopener noreferrer"> https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas.</a>


Import the watsonx.ai client library.

In [ ]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

<a id='create'></a>
### Create a client instance

Use your IBM Cloud API key. You can find information on how to get your API key <a href="https://dataplatform.cloud.ibm.com/docs/content/DO/WML_Deployment/DeployModelRest.html?audience=wdp&context=cpdaas#tasktask_deploymodelREST__prereq_el2_nft_bhb">here</a> and the instance URL <a href="https://cloud.ibm.com/apidocs/machine-learning#endpoint-url">here</a>.

In [ ]:
# Instantiate a client using credentials
credentials = Credentials(
      api_key = "<API_key>",
      url = "<instance_url>"
)

client = APIClient(credentials)

In [ ]:
client.version

<a id='prepare'></a>
### Prepare your model archive

Use the `write_file` command to write the model to a `burger.py` and `burger.mst` files. 

Use the `tar` command to create a tar archive.

In [ ]:
%mkdir model

You have to use the docplex.util.environment get_input_stream method to get access to the MIP start file from the docplex python script.

In [ ]:
%%writefile burger.py

import os
from docplex.mp.model import Model
from docplex.util.environment import get_environment

def apply_mip_starts(mdl):
    env = get_environment()
    with env.get_input_stream("burger.mst") as in_stream:
        path = os.path.realpath(in_stream.name)
        mdl.read_mip_starts(path)

def build_good_burger(**kwargs):
    mdl = Model("good_burger", **kwargs)

    items = ['beef', 'bun', 'cheese', 'onions', 'pickles', 'lettuce', 'ketchup', 'tomato']
    item_price = [0.25, 0.15, 0.1, 0.09, 0.03, 0.04, 0.02, 0.04]
    item_sodium = [50, 330, 310, 1, 260, 3, 160, 3]
    item_fat = [17, 9, 6, 2, 0, 0, 0, 0]
    item_calories = [220, 260, 70, 10, 5, 4, 20, 9]

    item_vars = mdl.integer_var_dict(items, name="how_many")

    for i in range(len(items)):
        mdl.add_constraint(item_vars[items[i]] <= 5)
        mdl.add_constraint(item_vars[items[i]] >= 1)

    mdl.add_constraint(mdl.sum(item_vars[items[h]] * item_sodium[h] for h in range(len(items))) <= 3000 - 1)
    mdl.add_constraint(mdl.sum(item_vars[items[h]] * item_fat[h] for h in range(len(items))) <= 150 - 1)
    mdl.add_constraint(mdl.sum(item_vars[items[h]] * item_calories[h] for h in range(len(items))) <= 3000 - 1)

    mdl.add_constraint(item_vars['ketchup'] == item_vars['lettuce'])
    mdl.add_constraint(item_vars['pickles'] == item_vars['tomato'])

    total_price = mdl.sum(item_vars[items[h]] * item_price[h] for h in range(len(items)))
    mdl.maximize(total_price)

    mdl.print_information()
    return mdl

mdl = build_good_burger()
apply_mip_starts(mdl)
s = mdl.solve(log_output=True)
if not s:
    print("BURGER model fails")
else:
    print('The price of the most expensive burger is ${}'.format(mdl.objective_value))
    #mdl.print_solution()

Partial MIP starts for this model.

In [ ]:
%%writefile burger.mst

<CPLEXSolution version="1.0">
 <header problemName="good_burger"/>
 <variables>
  <variable name="how_many_beef" index="0" value="5"/>
  <variable name="how_many_bun" index="1" value="5"/>
  <variable name="how_many_cheese" index="2" value="1"/>
  <variable name="how_many_onions" index="3" value="5"/>
 </variables>
</CPLEXSolution>

In [ ]:

import tarfile
def reset(tarinfo):
    tarinfo.uid = tarinfo.gid = 0
    tarinfo.uname = tarinfo.gname = "root"
    return tarinfo
tar = tarfile.open("model.tar.gz", "w:gz")
tar.add("burger.py", arcname="burger.py", filter=reset)
tar.add("burger.mst", arcname="burger.mst", filter=reset)
tar.close()


<a id='upload'></a>
### Upload your model on watsonx.ai Runtime

Store model in watsonx.ai Runtime with:
* the tar archive previously created,
* metadata including the model type and runtime

Get the `model_uid`.

In [ ]:
# Find the space ID

space_name = "<space_name>"

space_id = [x['metadata']['id'] for x in client.spaces.get_details()['resources'] if x['entity']['name'] == space_name][0]

client = APIClient(credentials, space_id = space_id)

In [ ]:
mnist_metadata = {
    client.repository.ModelMetaNames.NAME: "BurgerProduction",
    client.repository.ModelMetaNames.DESCRIPTION: "Model for Burger Production",
    client.repository.ModelMetaNames.TYPE: "do-docplex_22.1",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: client.software_specifications.get_id_by_name("do_22.1"),
}

model_details = client.repository.store_model(model='/home/wsuser/work/model.tar.gz', meta_props=mnist_metadata)
#model='/home/wsuser/work/model.tar.gz', 
model_uid = client.repository.get_model_id(model_details)

<a id='deploy'></a>
### Create a deployment 

Create a batch deployment for the model, providing information such as:
* the maximum number of compute nodes
* the T-shirt size of the compute nodes

Get the `deployment_uid`.

In [ ]:
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "BurgerProduction Deployment",
    client.deployments.ConfigurationMetaNames.DESCRIPTION: "BurgerProduction Deployment",
    client.deployments.ConfigurationMetaNames.BATCH: {},
    client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {'name': 'S', 'num_nodes': 1}
}

deployment_details = client.deployments.create(model_uid, meta_props=meta_props)

deployment_uid = client.deployments.get_id(deployment_details)

# print deployment id if needed
# print( deployment_uid )

In [ ]:
# List all existing deployments

client.deployments.list()

<a id='job'></a>
### Create and monitor a job with inline data for your deployed model

Create a payload containing inline input data.

Create a new job with this payload and the deployment.

Get the `job_uid`.

In [ ]:
solve_payload = {
    "solve_parameters" : {
        "oaas.logAttachmentName":"log.txt",
        "oaas.logTailEnabled":"true",
        "oaas.resultsFormat": "XML"
    },
    client.deployments.DecisionOptimizationMetaNames.INPUT_DATA: [
    ],
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA: [
        {
            "id":".*\\.xml"
        },
        {
            "id":"log.txt"
        }
    ]
}
job_details = client.deployments.create_job(deployment_uid, solve_payload)
job_uid = client.deployments.get_job_id(job_details)

Display job status until it is completed.

The first job of a new deployment might take some time as a compute node must be started.

In [ ]:
from time import sleep

while job_details['entity']['decision_optimization']['status']['state'] not in ['completed', 'failed', 'canceled']:
    print(job_details['entity']['decision_optimization']['status']['state'] + '...')
    sleep(5)
    job_details=client.deployments.get_job_details(job_uid)

print(job_details['entity']['decision_optimization']['solve_state']['solve_status'])

#### Check the MIP Starts were used by the engine

In [ ]:
import base64
import io

output_data = job_details['entity']['decision_optimization']['output_data']

starts_in_logs = [line for o in output_data if o['id'] == 'log.txt' for line in io.BytesIO(base64.b64decode(o['content'])) if "start" in str(line) ]
if len(starts_in_logs) == 0:
    print("Something went wrong")
elif len(starts_in_logs) == 1:
    print("MIP starts were provided to the job but engine rejected them")
    print("Something went wrong")
elif len(starts_in_logs) == 2:
    print("MIP starts were provided to the job and were used by the engine")
    for d in starts_in_logs:
        print(d)
else:
    print("Something went wrong")

### Delete the deployment

Use the following method to delete the deployment.

In [ ]:
client.deployments.delete(deployment_uid)

<a id='summary'></a>
### Summary and next steps

You've successfully completed this notebook! 

You've learned how to:

- work with the watsonx.ai client
- prepare your model archive and upload your model on watsonx.ai Runtime
- create a deployment
- create and monitor a job with inline data for your deployed model

Check out our online documentation for more samples, tutorials and documentation:
* <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=cpdaas" target="_blank" rel="noopener noreferrer">IBM Cloud Pak for Data as a Service documentation</a>
* <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx" target="_blank" rel="noopener noreferrer">IBM watsonx.ai documentation</a>

**Note**
In this sample, the optimization content (Python file + MIP Starts file) was uploaded as watsonx.ai Runtime model content (call to store_model). When the job is triggered, this model content is used: there is no need for any additional input data.

Another possible implementation could be to create an empty watsonx.ai Runtime model, then when triggering the job, pass the optimization content (Python file + MIP Starts file) as the job input data (client.deployments.DecisionOptimizationMetaNames.INPUT_DATA).

<hr>
Copyright © 2019-2026. This notebook and its source code are released under the terms of the MIT License.